# N132 · 组合博弈、Nim与SG函数

**目标：** 在无偏正常玩法条件下用状态异或合并游戏。

**先修：** N115, N125, N082。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 必胜/必败；mex；SG递推；Nim和；正常/反常玩法边界。

## 从问题到算法

无偏组合博弈中，双方可用动作只依赖当前状态，正常规则下无步可走者输。SG值是后继SG集合的最小缺失非负数mex；多个独立子游戏的SG值按异或合并。注意力扣can_win_nim接口是每次取1–3根的减法游戏，不能把其n%4规则误套到所有Nim变体。

## 最小实现

**本章接口：** `can_win_nim(n)`、`winner_square_game(n)`、`grundy(state, moves)`

In [1]:
from math import isqrt

def can_win_nim(n):
    if n<0: raise ValueError('nonnegative pile required')
    return n%4!=0

def winner_square_game(n):
    if n<0: raise ValueError('nonnegative pile required')
    dp=[False]*(n+1)
    for x in range(1,n+1): dp[x]=any(not dp[x-s*s] for s in range(1,isqrt(x)+1))
    return dp[n]

def grundy(state,moves):
    memo={}; active=set()
    def solve(s):
        if s in memo: return memo[s]
        if s in active: raise ValueError('finite acyclic game graph required')
        active.add(s); reachable={solve(t) for t in moves(s)}; active.remove(s)
        value=0
        while value in reachable: value+=1
        memo[s]=value; return value
    return solve(state)

## 正确性、前提与复杂度

SG为0时，所有后继SG都非零；SG非零时，mex定义保证存在SG为0的后继，因此输赢与SG是否为0一致。独立游戏和的异或为0时任一单游戏变化使异或非零；异或非零时可选最高不同位对应的分量转移到较小的指定SG，使总异或变0。该结论依赖正常玩法、无偏与有限终止。

**代价：** 取1..3 O(1)算术；平方取石O(n√n)时间O(n)空间；一般SG按可达DAG求值，O(V+E)级集合操作和O(V+E)空间上界。递归版仅用于小有限状态；有环游戏、失手规则和双方动作不同的游戏不能直接套此模板。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

moves=lambda n:[n-k for k in (1,2) if k<=n]
show_table(['石子数','取1或2的SG','取1..3是否必胜','取平方数是否必胜'],[(n,grundy(n,moves),can_win_nim(n),winner_square_game(n)) for n in range(11)])

石子数,取1或2的SG,取1..3是否必胜,取平方数是否必胜
0,0,False,False
1,1,True,True
2,2,True,False
3,0,True,True
4,1,False,True
5,2,True,False
6,0,True,True
7,1,True,False
8,2,False,True
9,0,True,True


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert not can_win_nim(4) and can_win_nim(5)
assert not winner_square_game(2)
assert grundy(0,lambda s:[])==0
from functools import cache
moves=lambda n:[n-k for k in (1,2) if k<=n]
@cache
def win_pair(a,b):
    return any(not win_pair(a-k,b) for k in (1,2) if a>=k) or any(not win_pair(a,b-k) for k in (1,2) if b>=k)
for a in range(9):
    for b in range(9): assert bool(grundy(a,moves)^grundy(b,moves))==win_pair(a,b)
@cache
def square_ref(n): return any(not square_ref(n-k*k) for k in range(1,isqrt(n)+1))
for n in range(80): assert winner_square_game(n)==square_ref(n)
try: grundy(0,lambda s:[s])
except ValueError: pass
else: raise AssertionError('cycle should not be silently treated as a losing state')
print('N132: 所有本章断言通过')

N132: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 用小状态完整搜索验证Nim规律。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 给出不满足无偏条件不能直接异或的游戏。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 292. Nim Game（canonical）
- 1510. Stone Game IV（canonical）
- 810. Chalkboard XOR Game（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。